In [0]:
from pyspark.sql.functions import *

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS fraud_platform.silver;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS fraud_platform.silver.silver_volume;

In [0]:
%sql
SHOW VOLUMES IN fraud_platform.silver;

database,volume_name
silver,silver_volume
silver,streaming_checkpoints


In [0]:
bronze_df = (
    spark.readStream
    .table("fraud_platform.bronze.transactions_raw")
)

In [0]:
print(bronze_df.isStreaming)

True


In [0]:
silver_df = (
    bronze_df
    .filter(col("TransactionID").isNotNull())
    .filter(col("TransactionAmt").isNotNull())
    .withColumn("ProductCD", upper(trim(col("ProductCD"))))
    .withColumn("card4", lower(trim(col("card4"))))
    .withColumn("card6", lower(trim(col("card6"))))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("event_date", current_date())
)

In [0]:
query = (
    silver_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/fraud_platform/silver/silver_volume/checkpoints/transactions_clean"
    )
    .toTable("fraud_platform.silver.transactions_clean")
)

query.awaitTermination()
print("Silver completed")

Silver completed


In [0]:
print(bronze_df.isStreaming)
print(silver_df.isStreaming)

True
True


In [0]:
for q in spark.streams.active:
    print(q.id)
    q.stop()

In [0]:
%sql
SELECT COUNT(*) FROM fraud_platform.bronze.transactions_raw;

COUNT(*)
20000


In [0]:
%sql
SELECT COUNT(*) FROM fraud_platform.silver.transactions_clean;

COUNT(*)
30000


In [0]:
%sql
SELECT *
FROM fraud_platform.silver.transactions_clean
LIMIT 10;

TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card4,card6,addr1,addr2,P_emaildomain,R_emaildomain,ingestion_timestamp,event_date
2987001,0,86401,29.0,W,mastercard,credit,325.0,87.0,gmail.com,null,2026-07-04T14:26:07.698Z,2026-07-04
2987003,0,86499,50.0,W,mastercard,debit,476.0,87.0,yahoo.com,null,2026-07-04T14:26:07.698Z,2026-07-04
2987005,0,86510,49.0,W,visa,debit,272.0,87.0,gmail.com,null,2026-07-04T14:26:07.698Z,2026-07-04
2987007,0,86529,422.5,W,visa,debit,325.0,87.0,mail.com,null,2026-07-04T14:26:07.698Z,2026-07-04
2987011,0,86555,16.495,C,mastercard,debit,null,null,hotmail.com,hotmail.com,2026-07-04T14:26:07.698Z,2026-07-04
2987013,0,86585,40.0,W,visa,debit,330.0,87.0,aol.com,null,2026-07-04T14:26:07.698Z,2026-07-04
2987016,0,86620,30.0,H,visa,debit,170.0,87.0,aol.com,null,2026-07-04T14:26:07.698Z,2026-07-04
2987018,0,86725,47.95,W,visa,debit,184.0,87.0,gmail.com,null,2026-07-04T14:26:07.698Z,2026-07-04
2987021,0,86769,159.95,W,mastercard,debit,204.0,87.0,gmail.com,null,2026-07-04T14:26:07.698Z,2026-07-04
2987024,0,86821,73.95,W,visa,debit,264.0,87.0,gmail.com,null,2026-07-04T14:26:07.698Z,2026-07-04
